In [24]:
import numpy as np
import math

def NextRandLCG(state):
    state[0] = (1664525 * state[0] + 1013904223) & 0xFFFFFFFF
    return state[0]

def Matrix_Index2D(i, j, cols):
    return i * cols + j

def RandomMatrixLCG(rows, cols, seed, low, high):
    out = np.empty(rows * cols, dtype=np.int32)
    for i in range(rows):
        for j in range(cols):
            idx = Matrix_Index2D(i, j, cols)
            out[idx] = low + (NextRandLCG(seed) % (high - low + 1))
    return out.reshape(rows, cols)

def FindGramMatrix(basis):
    d, n = B.shape
    G = np.zeros((n, n), dtype=object)
    for j in range(n):
        for i in range(j + 1):
            s = sum(B[l, j] * B[l, i] for l in range(d))
            G[j, i] = G[i, j] = s
    return G

def FindNorm(B):
    return sum(int(x)**2 for x in B.flatten())

n_param = 0.51
delta = 0.99
nn = (n_param + 0.5) / 2.0
dd = (delta + 1.0) / 2.0

def int64L2(B):
    B = np.array(B, dtype=object)
    d, n = B.shape
    assert(d >= n)
    G = FindGramMatrix(B)
    print(f"Gram: {G}")
    squareNorm = np.zeros((n, n), dtype=float)
    gramCoefficients = np.zeros((n, n), dtype=float)
    squareNorm[0, 0] = float(G[0, 0])
    k = 1
    while k < n:
        for i in range(k + 1):
            squareNorm[i, k] = float(G[i, k])
            for j in range(i):
                squareNorm[i, k] -= squareNorm[j, k] * gramCoefficients[j, i]
            gramCoefficients[i, k] = squareNorm[i, k] / squareNorm[i, i]
        max_val = max(abs(gramCoefficients[j, k]) for j in range(k))
        if max_val > nn:
            for j in range(k - 1, -1, -1):
                X = math.floor(float(gramCoefficients[j, k]) + 0.5)
                if X:
                    for i in range(d):
                        B[i, k] -= X * B[i, j]
                    for i in range(n):
                        dot = sum(B[l, i] * B[l, k] for l in range(d))
                        G[k, i] = G[i, k] = dot
                    for i in range(j):
                        gramCoefficients[i, k] -= X * gramCoefficients[i, j]
            continue
        if dd * squareNorm[k-1, k-1] < squareNorm[k, k] + gramCoefficients[k-1, k]**2 * squareNorm[k-1, k-1]:
            k += 1
        else:
            B[:, [k-1, k]] = B[:, [k, k-1]]
            for m in [k-1, k]:
                for i in range(n):
                    dot = sum(B[l, i] * B[l, m] for l in range(d))
                    G[m, i] = G[i, m] = dot
            for col in [k-1, k]:
                for i in range(col + 1):
                    squareNorm[i, col] = float(G[i, col])
                    for j in range(i):
                        squareNorm[i, col] -= squareNorm[j, col] * gramCoefficients[j, i]
                    gramCoefficients[i, col] = squareNorm[i, col] / squareNorm[i, i]
            k -= 1
            if k < 1:
                k = 1
    return np.array(B, dtype=np.int64)



seed = [12345]
B = RandomMatrixLCG(6, 5, seed, 0, 10)

before = FindNorm(B)
R = int64L2(B)
after = FindNorm(R)

print("Original basis:")
print(B)

print("\nReduced basis:")
print(R)

print("\nSquared norm before:", before)
print("Squared norm after :", after)

print("\nReduction happened:", after < before)

Gram: [[275 149 160 92 263]
 [149 114 63 54 123]
 [160 63 195 120 194]
 [92 54 120 100 102]
 [263 123 194 102 279]]
Original basis:
[[ 8  4  0  0  6]
 [ 6  0  5  0  9]
 [10  9  2  2  7]
 [ 1  0  2  4  0]
 [ 7  1  9  4  8]
 [ 5  4  9  8  7]]

Reduced basis:
[[ 4 -2 -2  4  0]
 [ 1  3 -2  2  1]
 [ 1 -3 -3  2  3]
 [ 3 -1  1 -1  0]
 [ 1  1 -4 -5 -2]
 [ 0  2  1 -1  7]]

Squared norm before: 963
Squared norm after : 205

Reduction happened: True


In [5]:
import numpy as np
import math

def NextRandLCG(state):
    state[0] = (1664525 * state[0] + 1013904223) & 0xFFFFFFFF
    return state[0]

def Matrix_Index2D(i, j, cols):
    return i * cols + j

def RandomMatrixLCG(rows, cols, seed, low, high):
    out = np.empty(rows * cols, dtype=np.int32)
    for i in range(rows):
        for j in range(cols):
            idx = Matrix_Index2D(i, j, cols)
            out[idx] = low + (NextRandLCG(seed) % (high - low + 1))
    return out.reshape(rows, cols)

def FindGramMatrix(basis):
    d, n = B.shape
    G = np.zeros((n, n), dtype=object)
    for j in range(n):
        for i in range(j + 1):
            s = sum(B[l, j] * B[l, i] for l in range(d))
            G[j, i] = G[i, j] = s
    return G

def Init_SquareAndCoeffs(squareNorm, gramCoefficients, colIndex, G):
    for i in range(colIndex + 1):
        squareNorm[i, colIndex] = float(G[i, colIndex])
        for j in range(i):
            squareNorm[i, colIndex] -= squareNorm[j, colIndex] * gramCoefficients[j, i]
        gramCoefficients[i, colIndex] = squareNorm[i, colIndex] / squareNorm[i, i]
    largestCoeff = max(abs(gramCoefficients[j, colIndex]) for j in range(colIndex))
    return largestCoeff

def PerformSizeReduction(colIndex, rows, cols, gramCoefficients, gramMatrix, basis):
    for j in range(colIndex - 1, -1, -1):
        #Find closest integer 
        X = math.floor(float(gramCoefficients[j, colIndex]) + 0.5)
        if X != 0:
            #Update basis
            for i in range(rows):
                basis[i, colIndex] -= X * basis[i, j]
            #Update gram matrix
            for i in range(cols):
                dot = sum(basis[l, i] * basis[l, colIndex] for l in range(rows))
                gramMatrix[colIndex, i] = gramMatrix[i, colIndex] = dot
            #Update gram coefficients
            for i in range(j):
                gramCoefficients[i, colIndex] -= X * gramCoefficients[i, j]    
                
def FindNorm(B):
    return sum(int(x)**2 for x in B.flatten())

n_param = 0.51
delta = 0.99
nn = (n_param + 0.5) / 2.0
dd = (delta + 1.0) / 2.0

def L2Reduction(B):
    B = np.array(B, dtype=object)
    rows, cols = B.shape
    assert(rows >= cols)
    G = FindGramMatrix(B)
    print(f"Gram: {G}")
    squareNorm = np.zeros((cols, cols), dtype=float)
    gramCoefficients = np.zeros((cols, cols), dtype=float)
    squareNorm[0, 0] = float(G[0, 0])
    currentCol = 1
    
    while currentCol < cols:
        largestCoeff = Init_SquareAndCoeffs(squareNorm, gramCoefficients, currentCol, G)
        #print(f"k: {k} largestCoeff: {largestCoeff}\nsquareNorm: {squareNorm}\n")
        #break
        if largestCoeff > nn:
            PerformSizeReduction(currentCol, rows, cols, gramCoefficients, G, B)
            continue
        if dd * squareNorm[currentCol-1, currentCol-1] < squareNorm[currentCol, currentCol] + gramCoefficients[currentCol-1, currentCol]**2 * squareNorm[currentCol-1, currentCol-1]:
            currentCol += 1
        else:
            #Swap columns
            B[:, [currentCol-1, currentCol]] = B[:, [currentCol, currentCol-1]]
            #Recompute Gram schmidt values after swap
            for m in [currentCol-1, currentCol]:
                for i in range(cols):
                    dot = sum(B[l, i] * B[l, m] for l in range(rows))
                    G[m, i] = G[i, m] = dot

            for col in [currentCol-1, currentCol]:
                for i in range(col + 1):
                    squareNorm[i, col] = float(G[i, col])
                    for j in range(i):
                        squareNorm[i, col] -= squareNorm[j, col] * gramCoefficients[j, i]
                    gramCoefficients[i, col] = squareNorm[i, col] / squareNorm[i, i]
            #Decrease to previous
            currentCol -= 1
            if currentCol < 1:
                currentCol = 1
    return np.array(B, dtype=np.int64)

seed = [12345]
B = RandomMatrixLCG(6, 5, seed, 0, 10)

before = FindNorm(B)
R = L2Reduction(B)
after = FindNorm(R)

print("Original basis:")
print(B)

print("\nReduced basis:")
print(R)

print("\nSquared norm before:", before)
print("Squared norm after :", after)

print("\nReduction happened:", after < before)

Gram: [[275 149 160 92 263]
 [149 114 63 54 123]
 [160 63 195 120 194]
 [92 54 120 100 102]
 [263 123 194 102 279]]
Original basis:
[[ 8  4  0  0  6]
 [ 6  0  5  0  9]
 [10  9  2  2  7]
 [ 1  0  2  4  0]
 [ 7  1  9  4  8]
 [ 5  4  9  8  7]]

Reduced basis:
[[ 4 -2 -2  4  0]
 [ 1  3 -2  2  1]
 [ 1 -3 -3  2  3]
 [ 3 -1  1 -1  0]
 [ 1  1 -4 -5 -2]
 [ 0  2  1 -1  7]]

Squared norm before: 963
Squared norm after : 205

Reduction happened: True
